In [4]:
# =========================================================
#  AI Agent Prototype: Crop Recommendation (Planner + Tools)
# =========================================================
print("\n=== AI Agent Prototype: Smart Crop Planner ===")

import pandas as pd
import re
import random

# --- a) Create a small internal dataset / knowledge base (mock)
crop_catalog = pd.DataFrame({
    "soil_type": ["Loamy", "Clay", "Sandy", "Loamy", "Clay", "Sandy"],
    "previous_crop": ["Rice", "Wheat", "Maize", "Cotton", "Groundnut", "Pulses"],
    "temperature": [28, 24, 30, 32, 27, 35],
    "rainfall": [220, 180, 120, 160, 140, 100],
    "ph": [6.5, 7.0, 6.8, 7.2, 6.0, 7.5],
    "recommended_crop": ["Wheat", "Rice", "Groundnut", "Maize", "Cotton", "Millet"]
})

crop_catalog.to_csv("agentic_crop_catalog.csv", index=False)
print("✅ Knowledge base ready (crop_catalog).")

# --- b) Define “tools” ---
def tool_lookup_conditions(area, soil_type, previous_crop):
    df = crop_catalog[(crop_catalog["soil_type"].str.lower() == soil_type.lower()) &
                      (crop_catalog["previous_crop"].str.lower() == previous_crop.lower())]
    if not df.empty:
        return df.iloc[0].to_dict()
    else:
        return {
            "soil_type": soil_type,
            "previous_crop": previous_crop,
            "temperature": random.randint(25, 35),
            "rainfall": random.randint(100, 250),
            "ph": round(random.uniform(6.0, 7.5), 1),
            "recommended_crop": random.choice(["Rice", "Wheat", "Maize", "Cotton", "Groundnut", "Millet"])
        }

def tool_predict_crop(temp, rain, ph, soil_type):
    if soil_type.lower() == "loamy" and 25 < temp < 30 and 150 < rain < 250:
        return "Wheat"
    elif soil_type.lower() == "clay" and ph < 7 and rain > 160:
        return "Rice"
    elif soil_type.lower() == "sandy" and ph > 6.5 and rain < 150:
        return "Groundnut"
    elif temp > 32 and rain < 150:
        return "Millet"
    else:
        return "Maize"

def tool_draft_report(area, soil_type, previous_crop, results):
    return f"""
📍 **Crop Advisory Report**
Region: {area}
Soil Type: {soil_type}
Previous Crop: {previous_crop}

🌡️ Estimated Temperature: {results['temperature']}°C  
🌧️ Estimated Rainfall: {results['rainfall']} mm  
⚗️ Soil pH: {results['ph']}

🌾 **Recommended Crop for Next Season:** {results['recommended_crop']}
""".strip()

# --- c) Planner (THINK → ACT → OBSERVE) ---
def ai_crop_planner(user_goal):
    print("Goal:", user_goal)
    
    # ✅ Robust region extraction (works with comma, period, or no punctuation)
    m_area = re.search(r"in\s+([A-Za-z\s]+?)(?:,|\sand|\.\s|$)", user_goal, re.IGNORECASE)
    area = m_area.group(1).strip() if m_area else "Unknown"
    
    # ✅ Debug print to verify parsing
    print(f"DEBUG: Extracted area → '{area}'")

    m_soil = re.search(r"soil\s*type\s*is\s*([A-Za-z]+)", user_goal, re.IGNORECASE)
    soil_type = m_soil.group(1).capitalize() if m_soil else "Loamy"
    
    m_prev = re.search(r"planted\s*([A-Za-z]+)", user_goal, re.IGNORECASE)
    previous_crop = m_prev.group(1).capitalize() if m_prev else "Rice"
    
    print(f"THINK: Fetching local conditions for {area} with {soil_type} soil and previous crop {previous_crop}.")
    
    conditions = tool_lookup_conditions(area, soil_type, previous_crop)
    print("OBSERVE: Found estimated conditions:")
    print(conditions)
    
    predicted_crop = tool_predict_crop(
        temp=conditions["temperature"],
        rain=conditions["rainfall"],
        ph=conditions["ph"],
        soil_type=soil_type
    )
    conditions["recommended_crop"] = predicted_crop
    print(f"THINK: Based on environmental fit, best crop → {predicted_crop}.")
    
    report = tool_draft_report(area, soil_type, previous_crop, conditions)
    print("OBSERVE: Drafted report ready.")
    
    return report

# --- d) Demo the agentic planner ---
goal_text = "In Andhra Pradesh, soil type is Loamy and we planted Rice last season. Suggest next crop."
result = ai_crop_planner(goal_text)
print("\nRESULT:\n")
print(result)



=== AI Agent Prototype: Smart Crop Planner ===
✅ Knowledge base ready (crop_catalog).
Goal: In Andhra Pradesh, soil type is Loamy and we planted Rice last season. Suggest next crop.
DEBUG: Extracted area → 'Andhra Pradesh'
THINK: Fetching local conditions for Andhra Pradesh with Loamy soil and previous crop Rice.
OBSERVE: Found estimated conditions:
{'soil_type': 'Loamy', 'previous_crop': 'Rice', 'temperature': 28, 'rainfall': 220, 'ph': 6.5, 'recommended_crop': 'Wheat'}
THINK: Based on environmental fit, best crop → Wheat.
OBSERVE: Drafted report ready.

RESULT:

📍 **Crop Advisory Report**
Region: Andhra Pradesh
Soil Type: Loamy
Previous Crop: Rice

🌡️ Estimated Temperature: 28°C  
🌧️ Estimated Rainfall: 220 mm  
⚗️ Soil pH: 6.5

🌾 **Recommended Crop for Next Season:** Wheat
